# Cats & Dogs ++ — Ensemble

Ensembling = **averaging predicted probabilities** of several trained CNNs (soft voting).
There is no meta-model, so this is not stacking.

We use checkpoints from the model notebook (all trained with validation on fold 0):
`run3_plateau`, `run4_lr3e-4` (EfficientNet-B0, linear head) and `run5_mlp_head` (EfficientNet-B0, MLP head).
The ensemble is chosen by CV on fold 0, not by the leaderboard.

## 1. Setup

In [1]:
!pip install -q timm iterative-stratification kagglehub

In [5]:
import os
import re
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import kagglehub
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/cats_dogs_pp"
CKPT_DIR = f"{ROOT}/checkpoints"
DEVICE = "cuda"

kagglehub.login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Kaggle credentials set.
Kaggle credentials successfully validated.


In [6]:

path = kagglehub.competition_download("cats-and-dogs-plus-plus")

LABELS = ["Cat", "Dog", "Moris", "Motya", "Biatrix"]
train_df = pd.read_csv(os.path.join(path, "train.csv")).merge(pd.read_csv(f"{ROOT}/folds.csv"), on="image_id")
test_df = pd.read_csv(os.path.join(path, "test.csv"))
val_df = train_df[train_df["fold"] == 0].reset_index(drop=True)
print("val:", len(val_df), "| test:", len(test_df))

100%|██████████| 1.66G/1.66G [00:33<00:00, 53.9MB/s]

Extracting files...


val: 2452 | test: 18390


Data for inference

In [7]:
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


class PetsDataset(Dataset):
    def __init__(self, df, folder):
        self.ids = df["image_id"].values
        self.folder = folder

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(path, self.folder, f"{self.ids[idx]}.jpg")).convert("RGB")
        return val_tf(img)

Load trained models

In [8]:
class PetsNet(nn.Module):
    def __init__(self, model_name, head):
        super().__init__()
        if head == "mlp":
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
            self.head = nn.Sequential(
                nn.Linear(self.backbone.num_features, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(512, len(LABELS)),
            )
        else:
            self.backbone = timm.create_model(model_name, pretrained=False, num_classes=len(LABELS))
            self.head = nn.Identity()

    def forward(self, x):
        return self.head(self.backbone(x))


def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    hp = ckpt["hyper_parameters"]
    model = PetsNet(hp["model_name"], hp.get("head", "linear"))
    state = {
        ("backbone." + k[len("model."):] if k.startswith("model.") else k): v
        for k, v in ckpt["state_dict"].items()
        if k.startswith(("model.", "backbone.", "head."))
    }
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


def best_ckpt(run_name):
    """Checkpoint of a run with the highest val_f1 in its file name."""
    files = glob.glob(os.path.join(CKPT_DIR, f"{run_name}-*.ckpt"))
    return max(files, key=lambda f: float(re.search(r"val_f1=(\d+\.\d+)", f).group(1)))


@torch.no_grad()
def predict(model, df, folder):
    loader = DataLoader(PetsDataset(df, folder), batch_size=64, shuffle=False, num_workers=2)
    return np.concatenate([torch.sigmoid(model(x.to(DEVICE))).cpu().numpy() for x in loader])

Predictions of each model (validation fold 0 and test)

In [9]:
RUNS = ["run3_plateau", "run4_lr3e-4", "run5_mlp_head"]

val_probs, test_probs = {}, {}
for run in RUNS:
    ckpt = best_ckpt(run)
    print(run, "->", os.path.basename(ckpt))
    model = load_model(ckpt)
    val_probs[run] = predict(model, val_df, "train")
    test_probs[run] = predict(model, test_df, "test")

run3_plateau -> run3_plateau-epoch=02-val_f1=0.8285.ckpt
run4_lr3e-4 -> run4_lr3e-4-epoch=02-val_f1=0.9084.ckpt
run5_mlp_head -> run5_mlp_head-epoch=03-val_f1=0.9450.ckpt


CV of single models and ensembles

In [10]:
y_val = val_df[LABELS].values


def f1_macro(probs, threshold=0.5):
    return f1_score(y_val, (probs > threshold).astype(int), average="macro", zero_division=0)


candidates = {run: [run] for run in RUNS}
candidates["ens_run4+run5"] = ["run4_lr3e-4", "run5_mlp_head"]
candidates["ens_run3+run4+run5"] = RUNS

results = pd.DataFrame([
    {"model": name, "members": ", ".join(members),
     "cv_f1": f1_macro(np.mean([val_probs[r] for r in members], axis=0))}
    for name, members in candidates.items()
]).sort_values("cv_f1", ascending=False).reset_index(drop=True)
display(results.round(4))

,model,members,cv_f1
0,ens_run4+run5,"run4_lr3e-4, run5_mlp_head",0.9481
1,run5_mlp_head,run5_mlp_head,0.9450
2,ens_run3+run4+run5,"run3_plateau, run4_lr3e-4, run5_mlp_head",0.9356
3,run4_lr3e-4,run4_lr3e-4,0.9084
4,run3_plateau,run3_plateau,0.8285


Submission of the best ensemble (chosen by CV)

In [11]:
best = results[results["model"].str.startswith("ens_")].iloc[0]
members = best["members"].split(", ")
print("chosen:", best["model"], "| CV F1 =", round(best["cv_f1"], 4))

ens_val = np.mean([val_probs[r] for r in members], axis=0)
per_label = f1_score(y_val, (ens_val > 0.5).astype(int), average=None, zero_division=0)
display(pd.DataFrame({"label": LABELS, "f1": per_label.round(3)}))

ens_test = np.mean([test_probs[r] for r in members], axis=0)
sub = pd.DataFrame((ens_test > 0.5).astype(int), columns=LABELS)
sub.insert(0, "image_id", test_df["image_id"].values)
out = f"{ROOT}/submissions/{best['model']}.csv"
sub.to_csv(out, index=False)
print("saved:", out, sub.shape)

chosen: ens_run4+run5 | CV F1 = 0.9481


,label,f1
0,Cat,0.989
1,Dog,0.986
2,Moris,1.000
3,Motya,0.824
4,Biatrix,0.941


saved: /content/drive/MyDrive/cats_dogs_pp/submissions/ens_run4+run5.csv (18390, 6)


Results

СV Public

run5 0.945 0.918
ensemble run4 + run5 0.948 0.910

The ensemble is slightly better on CV but worse on the leaderboard, so it did not help. run5 stays our best model.